# Twój generator 5 mln: analiza pliku partiami

Źródła: AI_analiza_danych_tools_11/DZIEN_1 (generator i analiza) oraz motyw agregacji z SANTANDER_ANALIZA_BIG_DATA/DZIEN_1/dask_python.ipynb. 35 minut. Standardowo 100 tys. pozycji i pandas bez Dask/Spark. Liczba wierszy sama nie definiuje Big Data. Ćwiczymy kontrolę zużycia pamięci i poprawność agregacji.



In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'dane').exists() and (ROOT.parent/'dane').exists(): ROOT=ROOT.parent
if not (ROOT/'dane').exists(): raise FileNotFoundError('Rozpakuj cały pakiet i otwórz notebook w folderze zawierającym dane oraz narzedzia.')
sys.path.insert(0,str(ROOT/'narzedzia'))
from swps_tools import sales_quality, aggregate_chunks, to_cents, AmountCapper, make_transactions

## 1. Format i próbka

Wczytaj pierwsze 5000 wierszy elektronika_100k.csv z właściwym separatorem, kodowaniem i przecinkiem dziesiętnym. Ustal jednostkę oraz wartość pozycji. Zapisz, które wielkości próbka opisuje dokładnie, a które tylko przybliża.

In [3]:
path=ROOT/'dane'/'elektronika_100k.csv'
sample = pd.read_csv(path, sep=';', encoding='utf-8-sig', decimal=',', nrows=5000)
sample.head()

,id,nazwa,ilość sztuk,cena netto,kolor,miasto,województwo,kategoria,wartość netto
0,1,Klawiatura biurowa,16,33.86,czarny,Warszawa,mazowieckie,Peryferia komputerowe,541.76
1,2,Karta graficzna premium,8,11255.19,zielony,Wrocław,dolnośląskie,Podzespoły komputerowe,90041.52
2,3,Zmywarka premium,6,3814.99,czerwony,Krosno,podkarpackie,Duże AGD,22889.94
3,4,Smartfon flagowy,12,5694.06,inne,Zamość,lubelskie,Urządzenia mobilne,68328.72
4,5,Soundbar premium,4,4525.89,żółty,Brzeg,opolskie,Audio i RTV,18103.56


In [5]:
print(f'Średnia cena w próbce: {sample['cena netto'].mean():.2f}')

Średnia cena w próbce: 3124.33


## 2. Agregacja całego pliku

Przeczytaj implementację aggregate_chunks w narzedzia/swps_tools.py. Policz cały plik w partiach 17003 rekordy. Zauważ, że ostatnia partia jest mniejsza. Podaj n, sumę sztuk, sumę wartości w groszach i średnią ceny jednostkowej. Nie konkatenować wszystkich partii.

In [6]:
result = aggregate_chunks(path, chunksize=17003)
print({k:v for k,v in result.items() if k!='groups'})
print(result['groups'].to_string())

{'rows': 100000, 'units': 1804250, 'net_cents': 558013808069, 'price_cents_sum': 30950504596, 'mean_unit_price_pln': 3095.0504596, 'quantity_weighted_price_pln': 3092.774327665235}
                              n   units     net_cents
Akcesoria elektroniczne    9338  168054    1684524485
Audio i RTV               12767  229698  113539620664
Duże AGD                  16698  299916  121884821579
Komputery i monitory      10914  197772  167768942662
Małe AGD                  22299  405496   61387314987
Peryferia komputerowe      7372  133255    4399866753
Podzespoły elektroniczne   1948   34868      15970446
Podzespoły komputerowe     3699   66618   29123484557
Sieci komputerowe          3732   67670    8404436481
Urządzenia mobilne        11233  200903   49804825455


## 3. Średnia partii

Policz średnie cen w partiach po 17003, a potem ich zwykłą średnią. Porównaj z sumą wszystkich cen dzieloną przez liczbę pozycji. Skąd może brać się różnica? Porównaj także średnią cen pozycji z ceną ważoną ilością sztuk.

In [9]:
means = []; counts = []; sums = []
for part in pd.read_csv(path, sep=';', encoding='utf-8-sig', chunksize=17003, usecols=['cena netto'], dtype=str):
    cents = to_cents(part['cena netto'])
    means.append(cents.mean()); counts.append(len(cents)); sums.append(cents.sum())
wrong = np.mean(means)/100
correct = sum(sums)/sum(counts)/100
print(f"rozmiar partii: {counts}")
print(f"średnia średnich: {wrong:.2f}, poprawna średnia pozycji: {correct}")
print(f"cena ważona sztukami: {result['quantity_weighted_price_pln']}")

rozmiar partii: [17003, 17003, 17003, 17003, 17003, 14985]
średnia średnich: 3095.08, poprawna średnia pozycji: 3095.0504596
cena ważona sztukami: 3092.774327665235


## 4. Niezależna kontrola

Dla dostarczonych 100 tys. pozycji możesz jednorazowo wczytać cały plik i potwierdzić sumy. To kontrola dydaktyczna na małym wariancie. Główna metoda z partii działa bez pełnego wczytania. Dla 5 mln użyj generatora i agregacji, nie włączaj poniższej kontroli bez oceny RAM.

In [12]:
reference = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str)
ref_net = to_cents(reference['wartość netto'])
ref_price = to_cents(reference['cena netto'])

print(ref_net, ref_price)

0          54176
1        9004152
2        2288994
3        6832872
4        1810356
          ...   
99995    3097325
99996     116184
99997    5247528
99998    8702484
99999       5088
Name: wartość netto, Length: 100000, dtype: int64 0           3386
1        1125519
2         381499
3         569406
4         452589
          ...   
99995     281575
99996      19364
99997     159016
99998     621606
99999        848
Name: cena netto, Length: 100000, dtype: int64


## Wariant 5 mln
W terminalu folderu pakietu: `python narzedzia/generuj_dane_swps.py --n 5000000 --output dane/elektronika_5mln.csv`. Potem `aggregate_chunks(ROOT / "dane/elektronika_5mln.csv", chunksize=100000)`. Oczekuj setek MB na dysku. Generator zapisuje partie; analiza utrzymuje pojedynczą partię i niewielki stan grup. Nie obiecujemy, że sam chunksize ogranicza pamięć wszystkich możliwych algorytmów. Dokładne globalne kwantyle i pełna walidacja są w oryginalnym analizatorze dołączonym w narzedzia.

In [13]:
!python generuj_dane_swps.py --n 5000000 --output dane/elektornika_5mln.csv

Zapis 5,000,000 wierszy do: /content/dane/elektornika_5mln.csv
500,000/5,000,000 (10.0%) | 192,842 wierszy/s
1,000,000/5,000,000 (20.0%) | 193,790 wierszy/s
1,500,000/5,000,000 (30.0%) | 192,093 wierszy/s
2,000,000/5,000,000 (40.0%) | 181,942 wierszy/s
2,500,000/5,000,000 (50.0%) | 182,558 wierszy/s
3,000,000/5,000,000 (60.0%) | 183,968 wierszy/s
3,500,000/5,000,000 (70.0%) | 184,712 wierszy/s
4,000,000/5,000,000 (80.0%) | 183,431 wierszy/s
4,500,000/5,000,000 (90.0%) | 179,288 wierszy/s
5,000,000/5,000,000 (100.0%) | 180,368 wierszy/s
Gotowe. Rozmiar: 452.0 MiB
Liczebności kolorów: czerwony: 1,250,000, zielony: 1,250,000, czarny: 1,000,000, żółty: 500,000, inne: 1,000,000
